In [1]:
# --- ETAPA 1: FILTRAGEM E PREPARAÇÃO DO PAINEL ESTADUAL (MINAS GERAIS) ---

import pandas as pd

# Defino o caminho da Base Mestra consolidada no pipeline anterior.
# Esta base contém todos os municípios brasileiros tratados e normalizados.
caminho_mestra = r'C:\Users\fabio\TCC\BASE_STATA_FINAL.csv'

print("--- Iniciando Preparação do Painel: Minas Gerais ---")

# Realizo o carregamento assegurando que os identificadores permaneçam como string.
df_mestra = pd.read_csv(caminho_mestra, dtype={'id_municipio_6': str, 'id_ano': str})

# FILTRAGEM GEOGRÁFICA: Isolo apenas os municípios do estado de Minas Gerais.
# Utilizo o prefixo '31' (IBGE) para extrair o conjunto de dados que servirá 
# tanto para a unidade de tratamento quanto para o pool de doadores do modelo.
df_minas_painel = df_mestra[df_mestra['id_municipio_6'].str.startswith('31')].copy()

print(f"Painel de Minas Gerais consolidado. Total de observações: {len(df_minas_painel)}")

# Validação rápida da estrutura do painel utilizando a unidade de tratamento (Araxá).
print("\n--- Verificação de Estrutura (Exemplo: Araxá) ---")
print(df_minas_painel[df_minas_painel['id_municipio_6'] == '310400'].head(3))

--- ETAPA 1: CARREGAR E PREPARAR MINAS GERAIS (PAINEL) ---
Base carregada. Total de linhas de MG: 28149
Veja que as cidades continuam separadas (ex: Araxá):
      id_municipio_6 id_ano id_nome_municipio id_uf  bruto_populacao  \
75876         310400   1992             Araxá    MG          71317.0   
75877         310400   1993             Araxá    MG          72695.0   
75878         310400   1994             Araxá    MG          73959.0   

       bruto_vab_total  bruto_vab_industria  bruto_vab_servicos  \
75876              NaN                  NaN                 NaN   
75877              NaN                  NaN                 NaN   
75878              NaN                  NaN                 NaN   

       bruto_cfem_valor  bruto_arrecadacao_total  ...  bruto_emp_mineracao  \
75876               NaN                      NaN  ...                  NaN   
75877               NaN                      NaN  ...                  NaN   
75878               NaN                      NaN  .

In [2]:
# --- ETAPA 2: ENRIQUECIMENTO DO PAINEL (INDICADORES DE SAÚDE INFANTIL) ---

# Para uma análise multidimensional do impacto econômico, incorporo a variável 
# de baixo peso ao nascer, um indicador proxy fundamental para o bem-estar social.

# 1. CARREGAMENTO DOS DADOS DE SAÚDE
caminho_peso = r'C:\Users\fabio\TCC\FINALIZADOS\11_peso_FINAL.csv'
df_peso = pd.read_csv(caminho_peso, dtype={'id_municipio_6': str, 'id_ano': str})

print(f"Dataset de indicadores de saúde carregado: {len(df_peso)} observações.")

# 2. INTEGRAÇÃO VIA LEFT JOIN
# Realizo a junção (merge) utilizando a base de Minas Gerais como referência.
# O método 'left' garante a preservação da estrutura temporal do painel original, 
# integrando a métrica 'ind_baixo_peso_pct' apenas onde houver correspondência de chaves.
print("Executando merge de dados setoriais...")

df_minas_painel = pd.merge(
    df_minas_painel,
    df_peso[['id_municipio_6', 'id_ano', 'ind_baixo_peso_pct']], 
    on=['id_municipio_6', 'id_ano'],
    how='left'
)

# 3. VALIDAÇÃO DE INTEGRIDADE (UNIDADE DE TRATAMENTO)
# Verifico se a nova variável foi corretamente atribuída à série temporal de Araxá.
print("\n--- Auditoria de Variável: Baixo Peso ao Nascer (Araxá) ---")
cols_validacao = ['id_ano', 'ind_baixo_peso_pct']
print(df_minas_painel[df_minas_painel['id_municipio_6'] == '310400'][cols_validacao].tail())

Base de Peso carregada (21961 linhas).
Unindo tabelas...

Verificação (Araxá - Últimos 5 anos):
     id_ano  ind_baixo_peso_pct
1447   2020            8.932169
1448   2021           10.625000
1449   2022                 NaN
1450   2023                 NaN
1451   2024                 NaN


In [3]:
# --- ETAPA 3: INTEGRAÇÃO DE EXTERNALIDADES DE SAÚDE (SISTEMA RESPIRATÓRIO) ---

# Adiciono os dados de internações respiratórias para avaliar possíveis 
# externalidades da atividade minerária na saúde pública municipal.

# 1. CARREGAMENTO DOS DADOS SETORIAIS (SAÚDE/DATASUS)
caminho_resp = r'C:\Users\fabio\TCC\FINALIZADOS\13_Intern_Resp.csv'
df_resp = pd.read_csv(caminho_resp, dtype={'id_municipio_6': str, 'id_ano': str})

print(f"Dataset de internações respiratórias carregado: {len(df_resp)} registros.")

# 2. CONSOLIDADAÇÃO VIA LEFT JOIN
# Integro a variável de internações ao painel de Minas Gerais, mantendo a 
# consistência das séries temporais já tratadas.
print("Unificando indicadores de saúde ao painel principal...")

df_minas_painel = pd.merge(
    df_minas_painel,
    df_resp[['id_municipio_6', 'id_ano', 'bruto_internacoes_resp']], 
    on=['id_municipio_6', 'id_ano'],
    how='left'
)

# 3. TRATAMENTO DE VALORES AUSENTES (MISSING VALUES)
# Opto pela imputação de valor zero para registros inexistentes, assumindo a 
# ausência de notificações em períodos específicos para garantir a continuidade 
# do modelo de Controle Sintético.
df_minas_painel['bruto_internacoes_resp'] = df_minas_painel['bruto_internacoes_resp'].fillna(0)

# 4. AUDITORIA DE VARIÁVEIS CONSOLIDADAS (ARAXÁ)
print("\n--- Validação Multidimensional: Saúde e Bem-estar (Araxá) ---")
cols_validacao = ['id_ano', 'ind_baixo_peso_pct', 'bruto_internacoes_resp']
print(df_minas_painel[df_minas_painel['id_municipio_6'] == '310400'][cols_validacao].tail())

Lendo internações de: C:\Users\fabio\TCC\FINALIZADOS\13_Intern_Resp.csv
Base de Internações carregada (23884 linhas).
Unindo tabelas...

Verificação Final (Araxá - Últimos 5 anos):
     id_ano  ind_baixo_peso_pct  bruto_internacoes_resp
1447   2020            8.932169                   323.0
1448   2021           10.625000                   357.0
1449   2022                 NaN                   546.0
1450   2023                 NaN                   637.0
1451   2024                 NaN                   797.0


In [4]:
# --- ETAPA 4: AUDITORIA E INVENTÁRIO DO PAINEL CONSOLIDADO (MINAS GERAIS) ---

# Realizo uma auditoria final para validar a integridade do painel estadual. 
# Este passo assegura que o enriquecimento com as variáveis de saúde não 
# comprometeu a estrutura do dataset e que os dados estão prontos para modelagem.

print(f"--- SUMÁRIO EXECUTIVO: PAINEL MINAS GERAIS ---")
print(f"Dimensão Espacial (Municípios): {df_minas_painel['id_municipio_6'].nunique()}")
print(f"Dimensão Temporal (Observações): {len(df_minas_painel)}")
print(f"Dimensão de Atributos (Variáveis): {len(df_minas_painel.columns)}")

print("\n--- DICIONÁRIO TÉCNICO E TIPAGEM (DTYPES) ---")
# Validação da tipagem: 'float64' para métricas contínuas e 'object' para identificadores.
print(df_minas_painel.dtypes)

print("\n--- DIAGNÓSTICO DE DADOS AUSENTES (MISSING VALUES) ---")
# Auditoria de integridade para as variáveis críticas de saúde.
# A ausência de nulos em 'Internações' confirma o sucesso da imputação anterior.
print("Lacunas em Baixo Peso ao Nascer:", df_minas_painel['ind_baixo_peso_pct'].isnull().sum())
print("Lacunas em Internações Resp.:", df_minas_painel['bruto_internacoes_resp'].isnull().sum())

# Exportação Final para Estimação no Stata
# path_final = r'C:\Users\fabio\TCC\BASE_MINAS_SAUDE_FINAL.dta'
# df_minas_painel.to_stata(path_final, write_index=False)

--- RESUMO DA BASE FINAL (Minas Gerais) ---
Total de Municípios: 853
Total de Linhas (Observações): 28149
Total de Colunas: 23

--- LISTA DE VARIÁVEIS E TIPOS ---
id_municipio_6                object
id_ano                        object
id_nome_municipio             object
id_uf                         object
bruto_populacao              float64
bruto_vab_total              float64
bruto_vab_industria          float64
bruto_vab_servicos           float64
bruto_cfem_valor             float64
bruto_arrecadacao_total      float64
bruto_emp_formal_total       float64
bruto_emp_mineracao          float64
bruto_emp_servicos           float64
bruto_pib_real               float64
ind_mortalidade_infantil     float64
ind_pib_pc_real_mil          float64
ind_part_vab_industria       float64
ind_part_emp_mineracao       float64
ind_cresc_pib_pc_real_mil    float64
ind_cresc_emp_total          float64
ind_dependencia_cfem         float64
ind_baixo_peso_pct           float64
bruto_internacoes_resp 

In [5]:
# --- ETAPA 5: EXPORTAÇÃO DEFINITIVA E COMPATIBILIDADE ---

# Nesta etapa final, exporto o painel consolidado de Minas Gerais. 
# Garanto a interoperabilidade entre Python e Stata, assegurando que 
# os tipos de dados e o horizonte temporal estejam otimizados para a análise.

# Defino caminhos estruturados para facilitar a gestão de versões do TCC.
path_csv = r'C:\Users\fabio\TCC\FINALIZADOS\BASE_MINAS_PAINEL_COMPLETA.csv'
path_dta = r'C:\Users\fabio\TCC\FINALIZADOS\BASE_MINAS_PAINEL_COMPLETA.dta'

print("Iniciando exportação dos datasets finais...")

# 1. ARQUIVAMENTO EM CSV
# Exportação para formato agnóstico, ideal para backups e auditoria externa.
df_minas_painel.to_csv(path_csv, index=False)

# 2. EXPORTAÇÃO OTIMIZADA PARA STATA (.DTA)
# Utilizo a versão 117 para garantir ampla compatibilidade com versões 13+ do Stata.
# O parâmetro write_index=False evita a poluição do dataset com índices do Pandas.
try:
    df_minas_painel.to_stata(path_dta, write_index=False, version=117)
    print(f"\n--- EXPORTAÇÃO CONCLUÍDA COM SUCESSO ---")
    print(f"Dataset de Análise (MG): {path_dta}")
except Exception as e:
    print(f"\n[ALERTA] Falha na gravação do arquivo .dta: {e}")
    print("Verifique se o arquivo está sendo utilizado por outro processo.")

print("\nStatus: Pipeline de preparação encerrado. O dataset está pronto para estimação econométrica.")

Salvando arquivos finais...

SUCESSO TOTAL! Base salva em:
-> C:\Users\fabio\TCC\FINALIZADOS\BASE_MINAS_PAINEL_COMPLETA.dta

Agora você pode abrir o Stata e carregar este arquivo!


In [6]:
# --- ETAPA EXTRA: NORMALIZAÇÃO DE INDICADORES (TAXA DE INTERNAÇÃO) ---

# Para garantir a comparabilidade entre municípios de diferentes portes populacionais, 
# calculo a taxa de internações respiratórias por 1.000 habitantes. 
# Este procedimento elimina o viés de escala e permite uma análise precisa 
# do impacto relativo na saúde pública.

print("Calculando indicadores de intensidade (Taxa por 1.000 hab)...")

# 1. CÁLCULO DO INDICADOR DE INTENSIDADE
# Fórmula: (Total de Internações / População Residente) * 1.000
df_minas_painel['ind_internacoes_resp_por_mil'] = (
    df_minas_painel['bruto_internacoes_resp'] / df_minas_painel['bruto_populacao']
) * 1000

# 2. TRATAMENTO DE CONSISTÊNCIA MATEMÁTICA
# Substituo eventuais valores infinitos (decorrentes de divisões por zero ou outliers) 
# por NaNs, assegurando a estabilidade das estimações no Stata.
import numpy as np
df_minas_painel['ind_internacoes_resp_por_mil'] = df_minas_painel['ind_internacoes_resp_por_mil'].replace([np.inf, -np.inf], np.nan)

# 3. VALIDAÇÃO TÉCNICA: ARAXÁ (BRUTO VS. TAXA)
# Verifico a evolução da taxa em relação ao número bruto para confirmar a tendência.
print("\n--- Auditoria de Normalização: Araxá (Série Recente) ---")
cols_validacao = ['id_ano', 'bruto_populacao', 'bruto_internacoes_resp', 'ind_internacoes_resp_por_mil']
print(df_minas_painel[df_minas_painel['id_municipio_6'] == '310400'][cols_validacao].tail())

# --- EXPORTAÇÃO DO DATASET DEFINITIVO (MODELAGEM) ---

path_csv = r'C:\Users\fabio\TCC\FINALIZADOS\BASE_MINAS_PAINEL_COMPLETA.csv'
path_dta = r'C:\Users\fabio\TCC\FINALIZADOS\BASE_MINAS_PAINEL_COMPLETA.dta'

print(f"\nConsolidando base final com indicadores normalizados...")
df_minas_painel.to_csv(path_csv, index=False)
df_minas_painel.to_stata(path_dta, write_index=False, version=117)

print("--- PIPELINE CONCLUÍDO ---")
print("Variável de controle/impacto disponível: 'ind_internacoes_resp_por_mil'")

Calculando taxa de internação...

Comparação Araxá (Bruto vs Taxa):
     id_ano  bruto_populacao  bruto_internacoes_resp  \
1447   2020         107337.0                   323.0   
1448   2021         108403.0                   357.0   
1449   2022              NaN                   546.0   
1450   2023              NaN                   637.0   
1451   2024         117677.0                   797.0   

      ind_internacoes_resp_por_mil  
1447                      3.009214  
1448                      3.293267  
1449                           NaN  
1450                           NaN  
1451                      6.772776  

Salvando base final com a nova taxa...
--- CONCLUÍDO! ---
Nova variável criada: 'ind_internacoes_resp_por_mil'


In [7]:
print(df_minas_painel.dtypes)

id_municipio_6                   object
id_ano                           object
id_nome_municipio                object
id_uf                            object
bruto_populacao                 float64
bruto_vab_total                 float64
bruto_vab_industria             float64
bruto_vab_servicos              float64
bruto_cfem_valor                float64
bruto_arrecadacao_total         float64
bruto_emp_formal_total          float64
bruto_emp_mineracao             float64
bruto_emp_servicos              float64
bruto_pib_real                  float64
ind_mortalidade_infantil        float64
ind_pib_pc_real_mil             float64
ind_part_vab_industria          float64
ind_part_emp_mineracao          float64
ind_cresc_pib_pc_real_mil       float64
ind_cresc_emp_total             float64
ind_dependencia_cfem            float64
ind_baixo_peso_pct              float64
bruto_internacoes_resp          float64
ind_internacoes_resp_por_mil    float64
dtype: object


In [ ]:
print(df_minas_painel.head)